# Phishing URL Detector — Full ML Pipeline
**Implementation:** Dataset → Feature Extraction → Preprocessing → Model Training → Evaluation → UI

---
**Model:** Random Forest Classifier (n_estimators=200)
**Dataset:** 1,600 URLs — 800 Phishing | 800 Legitimate


## Install Dependencies

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn joblib ipywidgets --quiet
print('All dependencies installed successfully!')

---
## STEP 1 — Dataset
A labelled dataset of 1,600 URLs (800 phishing, 800 legitimate) with 20 features + raw URL + label column.

In [ ]:
import pandas as pd
import numpy as np
import re
import math
from collections import Counter
from urllib.parse import urlparse

np.random.seed(42)

IP_PATTERN = re.compile(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$')
SHORTENERS = ['bit.ly', 'tinyurl.com', 'goo.gl', 'ow.ly', 't.co', 'is.gd', 'cutt.ly', 'rb.gy']
SUSPICIOUS_KEYWORDS = [
    'login', 'signin', 'verify', 'update', 'banking',
    'secure', 'account', 'confirm', 'password', 'suspend',
    'alert', 'support', 'validate', 'unlock', 'billing', 'webscr'
]
TARGET_BRANDS = [
    'google', 'netflix', 'facebook', 'microsoft', 'apple',
    'amazon', 'paypal', 'instagram', 'coinbase', 'dhl',
    'paytm', 'chase', 'sbi', 'irctc', 'wellsfargo',
    'fedex', 'binance', 'icici', 'hdfc', 'bankofamerica', 'whatsapp'
]

def sanitize_url(raw_url):
    url = raw_url.strip()
    if not (url.startswith('http://') or url.startswith('https://')):
        url = 'http://' + url
    return url

def extract_features(raw_url):
    sanitized = sanitize_url(raw_url)
    p = urlparse(sanitized)
    real_host = p.netloc.split('@')[-1].split(':')[0].strip().lower()
    is_ip = bool(IP_PATTERN.match(real_host))
    host_parts = real_host.split('.') if real_host else []
    url_length        = len(raw_url.strip())
    hostname_length   = len(real_host)
    path_length       = len(p.path)
    num_dots          = raw_url.count('.')
    num_hyphens       = raw_url.count('-')
    num_underscores   = raw_url.count('_')
    num_digits        = sum(c.isdigit() for c in raw_url)
    num_special_chars = sum(not c.isalnum() for c in raw_url)
    num_subdomains    = max(0, len(host_parts) - 2) if not is_ip else 0
    url_depth         = len([s for s in p.path.split('/') if s])
    num_query_params  = len(p.query.split('&')) if p.query else 0
    has_at_symbol     = 1 if '@' in raw_url else 0
    has_https         = 1 if p.scheme.lower() == 'https' else 0
    has_ip_address    = 1 if is_ip else 0
    target_short      = (real_host + p.path).lower()
    has_shortener     = 1 if any(s in target_short for s in SHORTENERS) else 0
    found_kw          = [kw for kw in SUSPICIOUS_KEYWORDS if kw in raw_url.lower()]
    num_suspicious_keywords = len(found_kw)
    has_brand = 0
    for b in TARGET_BRANDS:
        if b in raw_url.lower():
            if not (real_host == f'{b}.com' or real_host.endswith(f'.{b}.com')):
                has_brand = 1
                break
            if b in p.path.lower():
                has_brand = 1
                break
    digit_ratio = round(num_digits / url_length, 4) if url_length > 0 else 0.0
    prob        = [c / url_length for c in Counter(raw_url).values()] if url_length > 0 else [1]
    url_entropy = round(-sum(px * math.log2(px) for px in prob), 4)
    tld         = 'ip' if is_ip else (host_parts[-1] if host_parts else '')
    return {
        'url': raw_url,
        'url_length': url_length, 'hostname_length': hostname_length,
        'path_length': path_length, 'num_dots': num_dots,
        'num_hyphens': num_hyphens, 'num_underscores': num_underscores,
        'num_digits': num_digits, 'num_special_chars': num_special_chars,
        'num_subdomains': num_subdomains, 'url_depth': url_depth,
        'num_query_params': num_query_params, 'has_at_symbol': has_at_symbol,
        'has_https': has_https, 'has_ip_address': has_ip_address,
        'has_shortener': has_shortener,
        'num_suspicious_keywords': num_suspicious_keywords,
        'has_brand_in_subdomain_or_path': has_brand,
        'digit_ratio': digit_ratio, 'url_entropy': url_entropy,
        'tld': tld
    }

print('Feature extractor defined successfully!')

In [ ]:
# Seed URLs
legitimate_urls = [
    'https://www.google.com', 'https://www.youtube.com', 'https://www.github.com',
    'https://www.stackoverflow.com', 'https://www.wikipedia.org',
    'https://www.microsoft.com', 'https://www.apple.com', 'https://www.amazon.com',
    'https://www.netflix.com', 'https://www.twitter.com',
    'https://www.facebook.com', 'https://www.instagram.com',
    'https://www.linkedin.com', 'https://www.reddit.com',
    'https://www.twitch.tv', 'https://www.spotify.com',
    'https://www.adobe.com', 'https://www.dropbox.com',
    'https://www.slack.com', 'https://www.zoom.us',
    'https://www.chase.com', 'https://www.wellsfargo.com',
    'https://www.bankofamerica.com', 'https://www.citibank.com',
    'https://www.paypal.com', 'https://www.coinbase.com',
    'https://www.binance.com', 'https://www.robinhood.com',
    'https://www.sbi.co.in', 'https://www.icicibank.com',
    'https://www.hdfcbank.com', 'https://www.irctc.co.in',
    'https://www.paytm.com', 'https://www.flipkart.com',
    'https://www.myntra.com', 'https://www.zomato.com',
    'https://www.bbc.com', 'https://www.cnn.com',
    'https://www.theguardian.com', 'https://www.reuters.com',
    'https://www.nytimes.com', 'https://www.forbes.com',
    'https://www.ebay.com', 'https://www.etsy.com',
    'https://www.walmart.com', 'https://www.target.com',
    'https://www.bestbuy.com', 'https://www.aliexpress.com',
    'https://www.coursera.org', 'https://www.edx.org',
    'https://www.khanacademy.org', 'https://www.mit.edu',
    'https://www.harvard.edu', 'https://www.irs.gov',
    'https://www.usa.gov', 'https://www.cdc.gov',
    'https://cloud.google.com', 'https://www.heroku.com',
    'https://www.vercel.com', 'https://www.netlify.com',
    'https://www.digitalocean.com', 'https://www.cloudflare.com',
    'https://www.webmd.com', 'https://www.nih.gov',
    'https://www.who.int', 'https://www.weather.com',
    'https://www.imdb.com', 'https://www.yelp.com',
    'https://www.tripadvisor.com', 'https://www.booking.com',
    'https://www.airbnb.com', 'https://www.expedia.com'
]

phishing_urls = [
    'http://paypal-security-update.com/verify-account',
    'http://secure-login.apple.com.phishing-site.xyz/login',
    'http://192.168.1.1/phishing/bank-login.html',
    'http://login.microsoftonline.com.evil.ru/auth',
    'http://amazon-order-confirm.com/update-payment',
    'http://bit.ly/3xPhish1ng',
    'http://tinyurl.com/phish-link',
    'http://update-your-banking-details.tk/confirm',
    'http://10.0.0.1/admin/login.php',
    'http://secure-paypal-account@phishing.net/login',
    'http://facebook-secure-login.com/verify',
    'http://netflix-payment-failed.com/update-billing',
    'http://google-security-alert.tk/verify-account',
    'http://support-microsoft-help.com/windows-update',
    'http://irs-gov-tax-refund.com/claim-now',
    'http://instagram-account-suspended.ml/unlock',
    'http://whatsapp-prize-winner.ga/claim',
    'http://chase-bank-secure.info/signin',
    'http://hdfc-net-banking-secure.xyz/login',
    'http://sbi-online-banking-update.ga/verify',
    'http://confirm-your-paypal-account-immediately.com',
    'http://5.63.12.100/gate.php?id=victim@mail.com',
    'http://dropbox-shared-file.phishing.top/document.pdf',
    'http://coinbase-security.cc/verify-wallet',
    'http://wellsfargo-login.us.com/signin',
    'http://binance-airdrop.ml/claim-rewards',
    'http://login.ebay.com.fraudsite.info/signin',
    'http://amazon-gift-card-winner.xyz/redeem',
    'http://microsoft-support-alert.com/call-now',
    'http://apple-id-verify.ml/confirm-identity',
    'http://paytm-kyc-update.in.phish.top/verify',
    'http://icici-bank-secure.xyz/login',
    'http://irctc-login-secure.ml/book-ticket',
    'http://fedex-delivery-notification.com/track-package',
    'http://dhl-parcel-update.info/confirm-address',
    'http://linkedin-password-reset.com/reset',
    'http://twitter-account-verify.ml/confirm',
    'http://youtube-monetization-update.com/verify',
    'http://validate.account.chase-bank.phish.net',
    'http://bankofamerica.secure-login.xyz/signin'
]

def generate_synthetic_legitimate(n):
    base_domains = ['techblog','dailynews','shopstore','onlineservice','webportal',
                    'cloudservice','devtools','myaccount','userportal','datastore',
                    'apigateway','medianet','contentsite','healthportal','financeapp']
    tlds  = ['com','org','net','io','co','edu','gov','info']
    paths = ['/','/ about','/products','/services','/blog','/contact',
             '/faq','/help','/pricing','/home','/docs','/api']
    rng  = np.random.default_rng(42)
    urls = []
    for _ in range(n):
        dom    = rng.choice(base_domains)
        suffix = ''.join(rng.choice(list('abcdefghijklmnopqrstuvwxyz'), size=rng.integers(3,7)))
        tld    = rng.choice(tlds)
        path   = rng.choice(paths)
        urls.append(f"https://www.{dom}{suffix}.{tld}{path}")
    return urls

def generate_synthetic_phishing(n):
    patterns = [
        lambda rng: f"http://{rng.integers(1,255)}.{rng.integers(1,255)}.{rng.integers(1,255)}.{rng.integers(1,255)}/login.php",
        lambda rng: f"http://paypal-secure-{''.join(rng.choice(list('abcdefghijklmnopqrstuvwxyz'),size=5))}.tk/verify",
        lambda rng: f"http://www.apple.com.{''.join(rng.choice(list('abcdefghijklmnopqrstuvwxyz'),size=6))}.xyz/signin",
        lambda rng: f"http://bit.ly/{''.join(rng.choice(list('abcdefghijklmnopqrstuvwxyz0123456789'),size=7))}",
        lambda rng: f"http://secure-{''.join(rng.choice(list('abcdefghijklmnopqrstuvwxyz'),size=4))}-login.ga/verify",
        lambda rng: f"http://login.microsoft.com.{''.join(rng.choice(list('abcdefghijklmnopqrstuvwxyz'),size=5))}.ru/auth",
        lambda rng: f"http://{''.join(rng.choice(list('abcdefghijklmnopqrstuvwxyz'),size=5))}@banking.com/login",
        lambda rng: f"http://amazon-{''.join(rng.choice(list('abcdefghijklmnopqrstuvwxyz0123456789'),size=6))}.com/update-payment",
        lambda rng: f"http://netflix-{''.join(rng.choice(list('abcdefghijklmnopqrstuvwxyz0123456789'),size=5))}.ml/billing",
        lambda rng: f"http://{''.join(rng.choice(list('abcdefghijklmnopqrstuvwxyz0123456789-'),size=18))}.tk/suspend",
        lambda rng: f"http://support-verify-{''.join(rng.choice(list('abcdefghijklmnopqrstuvwxyz'),size=4))}.ml/confirm",
        lambda rng: f"http://tinyurl.com/{''.join(rng.choice(list('abcdefghijklmnopqrstuvwxyz0123456789'),size=8))}",
        lambda rng: f"http://hdfc-{''.join(rng.choice(list('abcdefghijklmnopqrstuvwxyz'),size=5))}-secure.xyz/login",
        lambda rng: f"http://sbi-net-banking-{''.join(rng.choice(list('abcdefghijklmnopqrstuvwxyz'),size=4))}.ga/verify",
        lambda rng: f"http://{''.join(rng.choice(list('abcdefghijklmnopqrstuvwxyz0123456789'),size=12))}.top/login?token={''.join(rng.choice(list('abcdefghijklmnopqrstuvwxyz0123456789'),size=10))}"
    ]
    rng  = np.random.default_rng(99)
    urls = []
    for _ in range(n):
        urls.append(rng.choice(patterns)(rng))
    return urls

all_legit    = (legitimate_urls + generate_synthetic_legitimate(800 - len(legitimate_urls)))[:800]
all_phishing = (phishing_urls   + generate_synthetic_phishing(800  - len(phishing_urls)))[:800]

records = []
for url in all_legit:
    r = extract_features(url); r['label'] = 0; records.append(r)
for url in all_phishing:
    r = extract_features(url); r['label'] = 1; records.append(r)

df = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)
df.to_csv('phishing_url_dataset.csv', index=False)

print('=' * 60)
print('  STEP 1 OUTPUT — Dataset Preview')
print('=' * 60)
print(f'  df.shape = {df.shape}')
print(f'  Columns  : {list(df.columns)}')
print(f'  Label Distribution:')
print(f'    Legitimate (0) = {(df["label"]==0).sum()}')
print(f'    Phishing   (1) = {(df["label"]==1).sum()}')
df.head(10)

---
## STEP 2 — Features Used

In [ ]:
feature_descriptions = {
    'url_length':                    'Total number of characters in the URL',
    'hostname_length':               'Length of the domain name',
    'path_length':                   'Length of the path after the domain',
    'num_dots':                      'Number of "." in the URL',
    'num_hyphens':                   'Number of "-" in the URL',
    'num_underscores':               'Number of "_" in the URL',
    'num_digits':                    'Number of digits in the URL',
    'num_special_chars':             'Number of non-alphanumeric characters',
    'num_subdomains':                'Number of subdomains',
    'url_depth':                     'Number of path levels',
    'num_query_params':              'Number of parameters after "?"',
    'has_at_symbol':                 '1 if "@" is present',
    'has_https':                     '1 if the URL uses HTTPS',
    'has_ip_address':                '1 if an IP address is used as the host',
    'has_shortener':                 '1 if a URL shortener is used (bit.ly, tinyurl, etc.)',
    'num_suspicious_keywords':       'Count of words like login, verify, secure, update',
    'has_brand_in_subdomain_or_path':'1 if a brand name appears outside the genuine domain',
    'digit_ratio':                   'Digits divided by total URL length',
    'url_entropy':                   'Randomness of characters (Shannon entropy)',
    'tld':                           'Top-level domain (.com, .xyz, .tk, etc.)'
}

feat_df = pd.DataFrame({'Feature': list(feature_descriptions.keys()),
                        'Description': list(feature_descriptions.values())})
feat_df.index = range(1, len(feat_df)+1)

print('=' * 60)
print('  STEP 2 OUTPUT — 20 Features')
print('=' * 60)
print(feat_df.to_string())
print()

numerical_features = [c for c in df.columns if c not in ['url','tld','label']]
print('\nDataset Statistics (numerical features):')
df[numerical_features].describe().round(2)

---
## STEP 3 — Pre-processing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print('=' * 60)
print('  STEP 3 OUTPUT — Pre-processing')
print('=' * 60)

X = df.drop(columns=['url', 'label'])
y = df['label']

categorical_cols = ['tld']
numerical_cols   = [c for c in X.columns if c not in categorical_cols]

print(f'  Feature matrix shape : X = {X.shape}')
print(f'  Target vector  shape : y = {y.shape}')
print(f'  Categorical columns  : {categorical_cols}')
print(f'  TLD unique values    : {sorted(X["tld"].unique().tolist())}')
print(f'  Numerical cols ({len(numerical_cols)})   : {numerical_cols}')

preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)],
    remainder='passthrough'
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'\n  Splitting 80:20 (stratified)...')
print(f'  Train set : {X_train.shape[0]} samples')
print(f'  Test set  : {X_test.shape[0]} samples')
print(f'  Train label dist: Legit(0)={int((y_train==0).sum())}  Phishing(1)={int((y_train==1).sum())}')
print(f'  Test  label dist: Legit(0)={int((y_test==0).sum())}   Phishing(1)={int((y_test==1).sum())}')

print('\n  Sample of X_train (first 5 rows):')
X_train.head()

---
## STEP 4 — Model Training (Random Forest, n_estimators=200)

In [ ]:
import joblib
from sklearn.ensemble import RandomForestClassifier

print('=' * 60)
print('  STEP 4 OUTPUT — Model Training')
print('=' * 60)

rf_model = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', rf_model)
])

print('  Pipeline:')
print('    Step 1: ColumnTransformer')
print('      - OneHotEncoder on tld column')
print('      - passthrough on 19 numerical columns')
print('    Step 2: RandomForestClassifier')
print('      - n_estimators = 200')
print('      - max_depth    = 15')
print('      - random_state = 42')
print()
print('  Training in progress...')
pipeline.fit(X_train, y_train)
print('  Training complete!')

# Feature importances
rf_clf  = pipeline.named_steps['classifier']
preproc = pipeline.named_steps['preprocessor']
ohe_cols = preproc.named_transformers_['cat'].get_feature_names_out(categorical_cols).tolist()
all_feature_names = ohe_cols + numerical_cols
importances = rf_clf.feature_importances_

fi_df = pd.DataFrame({'feature': all_feature_names, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).reset_index(drop=True)

print('\n  Top 10 Most Informative Features:')
print(f"  {'#':<4} {'Feature':<40} {'Importance':>10}")
print('  ' + '-'*55)
for i, row in fi_df.head(10).iterrows():
    bar = '|' * int(row['importance'] * 200)
    print(f"  {i+1:<4} {row['feature']:<40} {row['importance']*100:>8.2f}%  {bar}")

joblib.dump(pipeline, 'phishing_model.joblib')
print("\n  Model saved to 'phishing_model.joblib'")

---
## STEP 5 — Evaluation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
roc  = roc_auc_score(y_test, y_prob)
cm   = confusion_matrix(y_test, y_pred)

print('=' * 60)
print('  STEP 5 OUTPUT — Evaluation Metrics')
print('=' * 60)
print(f'  Accuracy   : {acc*100:.2f}%')
print(f'  Precision  : {prec:.4f}')
print(f'  Recall     : {rec:.4f}')
print(f'  F1-Score   : {f1:.4f}')
print(f'  ROC-AUC    : {roc:.4f}')
print()
print('  Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Legitimate','Phishing']))

In [ ]:
# Confusion Matrix + Feature Importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f1117')

ax1 = axes[0]
ax1.set_facecolor('#1a1d27')
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate','Phishing'],
            yticklabels=['Legitimate','Phishing'],
            ax=ax1, linewidths=0.5, linecolor='#2d3144',
            annot_kws={'size':18,'weight':'bold','color':'white'})
ax1.set_title('Confusion Matrix', color='white', fontsize=14, pad=12, fontweight='bold')
ax1.set_xlabel('Predicted Label', color='#aab2c8', fontsize=11)
ax1.set_ylabel('True Label', color='#aab2c8', fontsize=11)
ax1.tick_params(colors='#aab2c8')

ax2 = axes[1]
ax2.set_facecolor('#1a1d27')
colors = ['#4f8ef7' if i < 3 else '#6c757d' for i in range(10)]
bars = ax2.barh(fi_df['feature'].head(10)[::-1],
                fi_df['importance'].head(10)[::-1] * 100,
                color=colors[::-1], edgecolor='none', height=0.65)
for bar, val in zip(bars, fi_df['importance'].head(10)[::-1]*100):
    ax2.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2,
             f'{val:.2f}%', va='center', color='white', fontsize=8.5)
ax2.set_title('Top 10 Feature Importances', color='white', fontsize=14, pad=12, fontweight='bold')
ax2.set_xlabel('Importance (%)', color='#aab2c8', fontsize=10)
ax2.tick_params(colors='#aab2c8', labelsize=8)
ax2.spines[['top','right','bottom','left']].set_color('#2d3144')
ax2.set_xlim(0, fi_df['importance'].max()*120)

plt.suptitle('Phishing URL Detector — Model Performance', color='white',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117', edgecolor='none')
plt.show()
print("Plot saved to 'evaluation_results.png'")

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')
ax.plot(fpr, tpr, color='#4f8ef7', lw=2.5, label=f'ROC Curve (AUC = {roc:.4f})')
ax.plot([0,1],[0,1], color='#555', linestyle='--', lw=1.5, label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.15, color='#4f8ef7')
ax.set_xlabel('False Positive Rate', color='#aab2c8', fontsize=11)
ax.set_ylabel('True Positive Rate', color='#aab2c8', fontsize=11)
ax.set_title('ROC Curve — Random Forest Classifier', color='white', fontsize=13, fontweight='bold')
ax.legend(facecolor='#2d3144', edgecolor='#4f8ef7', labelcolor='white', fontsize=10)
ax.tick_params(colors='#aab2c8')
ax.spines[['top','right','bottom','left']].set_color('#2d3144')
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight', facecolor='#0f1117', edgecolor='none')
plt.show()
print("ROC curve saved to 'roc_curve.png'")

In [ ]:
# Metrics summary table
metrics_df = pd.DataFrame({
    'Metric':  ['Accuracy','Precision','Recall','F1-Score','ROC-AUC'],
    'Result':  [f'{acc*100:.2f}%', f'{prec:.4f}', f'{rec:.4f}', f'{f1:.4f}', f'{roc:.4f}'],
    'Interpretation': [
        'Overall correct predictions',
        'Of predicted phishing, how many are actually phishing',
        'Of actual phishing, how many did we catch',
        'Harmonic mean of Precision and Recall',
        'Area under ROC curve (1.0 = perfect)'
    ]
})
metrics_df.index = range(1, len(metrics_df)+1)
metrics_df

---
## STEP 6 — Interactive User Interface

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

display(HTML("""
<style>
.phish-card{background:linear-gradient(135deg,#1a1d27,#0f1117);border-radius:12px;padding:20px 28px;
  font-family:'Segoe UI',sans-serif;color:#e0e6f0;margin:8px 0;border:1px solid #2d3144;
  box-shadow:0 4px 24px rgba(0,0,0,0.4);}
.result-phishing{background:linear-gradient(135deg,#3b1212,#2a0808);border:2px solid #e05252;
  border-radius:10px;padding:16px 22px;margin-top:12px;}
.result-legit{background:linear-gradient(135deg,#0d2b1d,#061a10);border:2px solid #4caf88;
  border-radius:10px;padding:16px 22px;margin-top:12px;}
.verdict-text{font-size:22px;font-weight:700;margin-bottom:8px;}
.conf-bar-wrap{background:#2d3144;border-radius:6px;height:14px;margin:4px 0 10px 0;}
.conf-bar{height:14px;border-radius:6px;}
.feat-grid{display:grid;grid-template-columns:1fr 1fr;gap:6px 24px;margin-top:10px;font-size:13px;}
.feat-item{display:flex;justify-content:space-between;border-bottom:1px solid #2d3144;padding:3px 0;}
.feat-label{color:#8b95b0;}.feat-val{color:#e0e6f0;font-weight:600;}
.header{font-size:20px;font-weight:700;color:#4f8ef7;margin-bottom:4px;}
.sub{font-size:12px;color:#5c6480;margin-bottom:14px;}
.url-display{font-size:11.5px;color:#6a7490;word-break:break-all;margin-bottom:8px;}
</style>
"""))

url_input  = widgets.Text(placeholder='Enter URL (e.g. https://www.google.com)',
                           layout=widgets.Layout(width='600px', height='38px'))
btn_detect = widgets.Button(description='Detect', button_style='',
                             layout=widgets.Layout(width='100px', height='38px'))
btn_clear  = widgets.Button(description='Clear',  button_style='',
                             layout=widgets.Layout(width='80px',  height='38px'))
btn_detect.style.button_color = '#4f8ef7'
btn_clear.style.button_color  = '#2d3144'
output_area = widgets.Output()

def run_prediction(url_str):
    features = extract_features(url_str)
    feat_only = {k: v for k, v in features.items() if k != 'url'}
    X_input  = pd.DataFrame([feat_only])
    prediction = pipeline.predict(X_input)[0]
    proba      = pipeline.predict_proba(X_input)[0]
    phish_conf = proba[1]*100
    legit_conf = proba[0]*100

    if prediction == 1:
        vc, vi, vt = 'result-phishing', 'PHISHING DETECTED', '#e05252'
        bar_pct, bar_color = phish_conf, '#e05252'
        conf_label = f'Phishing Confidence: {phish_conf:.1f}%'
    else:
        vc, vi, vt = 'result-legit', 'LEGITIMATE', '#4caf88'
        bar_pct, bar_color = legit_conf, '#4caf88'
        conf_label = f'Legitimate Confidence: {legit_conf:.1f}%'

    signals = []
    if features.get('has_ip_address'):              signals.append('IP-based host')
    if features.get('has_at_symbol'):               signals.append('@-symbol in URL')
    if features.get('has_shortener'):               signals.append('URL Shortener')
    if features.get('num_suspicious_keywords',0)>0: signals.append(f"{features['num_suspicious_keywords']} suspicious keyword(s)")
    if features.get('has_brand_in_subdomain_or_path'):signals.append('Brand name in path/subdomain')
    if features.get('has_https')==0:                signals.append('No HTTPS')
    if features.get('num_subdomains',0)>2:          signals.append(f"{features['num_subdomains']} subdomains")

    sigs_html = ''.join(
        f'<span style="background:#3b2a0a;color:#f0a844;border-radius:4px;padding:2px 8px;margin:2px 3px;font-size:11px;display:inline-block;">! {s}</span>'
        for s in signals
    ) if signals else '<span style="color:#4caf88;font-size:12px;">No suspicious signals detected</span>'

    feat_items = [
        ('URL Length',         features.get('url_length',0)),
        ('Hostname Length',    features.get('hostname_length',0)),
        ('Path Length',        features.get('path_length',0)),
        ('Num Dots',           features.get('num_dots',0)),
        ('Num Hyphens',        features.get('num_hyphens',0)),
        ('Num Digits',         features.get('num_digits',0)),
        ('Num Special Chars',  features.get('num_special_chars',0)),
        ('Num Subdomains',     features.get('num_subdomains',0)),
        ('URL Depth',          features.get('url_depth',0)),
        ('Query Params',       features.get('num_query_params',0)),
        ('Has HTTPS',          'Yes' if features.get('has_https') else 'No'),
        ('Has @ Symbol',       'Yes' if features.get('has_at_symbol') else 'No'),
        ('Has IP Host',        'Yes' if features.get('has_ip_address') else 'No'),
        ('Has Shortener',      'Yes' if features.get('has_shortener') else 'No'),
        ('Suspicious KWs',     features.get('num_suspicious_keywords',0)),
        ('Brand Spoofed',      'Yes' if features.get('has_brand_in_subdomain_or_path') else 'No'),
        ('Digit Ratio',        f"{features.get('digit_ratio',0):.3f}"),
        ('URL Entropy',        f"{features.get('url_entropy',0):.3f}"),
        ('TLD',                features.get('tld','N/A')),
    ]
    feat_grid_html = ''.join(
        f'<div class="feat-item"><span class="feat-label">{k}</span><span class="feat-val">{v}</span></div>'
        for k,v in feat_items
    )

    return f"""
    <div class="phish-card">
      <div class="header">Phishing URL Detector</div>
      <div class="sub">Random Forest Classifier - n_estimators=200</div>
      <div class="url-display">Analysing: <b>{url_str[:100]}{'...' if len(url_str)>100 else ''}</b></div>
      <div class="{vc}">
        <div class="verdict-text" style="color:{vt};">{vi}</div>
        <div style="font-size:12px;color:#8b95b0;margin-bottom:6px;">{conf_label}</div>
        <div class="conf-bar-wrap"><div class="conf-bar" style="width:{bar_pct:.1f}%;background:{bar_color};"></div></div>
        <div style="display:flex;justify-content:space-between;font-size:11px;color:#6a7490;">
          <span>Legitimate: {legit_conf:.1f}%</span><span>Phishing: {phish_conf:.1f}%</span>
        </div>
      </div>
      <div style="margin-top:14px;">
        <div style="font-size:12px;font-weight:600;color:#8b95b0;margin-bottom:6px;">RISK SIGNALS</div>
        {sigs_html}
      </div>
      <div style="margin-top:14px;">
        <div style="font-size:12px;font-weight:600;color:#8b95b0;margin-bottom:6px;">EXTRACTED FEATURES (20 total)</div>
        <div class="feat-grid">{feat_grid_html}</div>
      </div>
    </div>"""

def on_detect(b):
    with output_area:
        clear_output(wait=True)
        url = url_input.value.strip()
        if not url:
            display(HTML('<div style="color:#e05252;padding:10px;">Please enter a URL first.</div>'))
            return
        try:
            display(HTML(run_prediction(url)))
        except Exception as e:
            display(HTML(f'<div style="color:#e05252;padding:10px;">Error: {e}</div>'))

def on_clear(b):
    with output_area: clear_output()
    url_input.value = ''

btn_detect.on_click(on_detect)
btn_clear.on_click(on_clear)

display(HTML("""
<div style="background:#0f1117;padding:18px 22px;border-radius:12px;border:1px solid #2d3144;margin-bottom:10px;">
  <div style="font-size:22px;font-weight:700;color:#4f8ef7;">Phishing URL Detector - Interactive UI</div>
  <div style="font-size:12px;color:#5c6480;">STEP 6 - Enter a URL and click Detect to classify it</div>
</div>
"""))
display(widgets.HBox([url_input, btn_detect, btn_clear],
                     layout=widgets.Layout(gap='8px', align_items='center')))
display(output_area)
print('UI ready! Enter a URL above and click Detect.')

In [ ]:
# STEP 6 — Batch Demo Predictions
print('=' * 60)
print('  STEP 6 — Batch Demo Predictions')
print('=' * 60)

demo_urls = [
    ('http://paypal-secure-login.tk/verify-account',          1, 'Phishing'),
    ('http://192.168.1.1/banking/login.html',                 1, 'Phishing'),
    ('http://secure-login.apple.com.phishing-site.xyz/login', 1, 'Phishing'),
    ('http://bit.ly/3xPhish1ng',                              1, 'Phishing'),
    ('http://facebook-account-verify.ga/login?user=admin',    1, 'Phishing'),
    ('https://www.google.com',                                0, 'Legitimate'),
    ('https://www.github.com/torvalds/linux',                 0, 'Legitimate'),
    ('https://stackoverflow.com/questions/12345',             0, 'Legitimate'),
    ('https://www.amazon.com/products/electronics',           0, 'Legitimate'),
    ('https://www.microsoft.com/en-us/windows',               0, 'Legitimate'),
]

print(f"\n  {'#':<3} {'URL':<52} {'Expected':<12} {'Predicted':<12} {'Phish%':>7} Match")
print('  ' + '-'*93)

correct = 0
for i, (url, exp_label, exp_name) in enumerate(demo_urls, 1):
    feat = extract_features(url)
    feat_only = {k:v for k,v in feat.items() if k != 'url'}
    X_inp = pd.DataFrame([feat_only])
    pred  = pipeline.predict(X_inp)[0]
    prob  = pipeline.predict_proba(X_inp)[0][1] * 100
    pred_name = 'Phishing' if pred==1 else 'Legitimate'
    match = 'OK' if pred==exp_label else 'MISS'
    if pred==exp_label: correct += 1
    short = (url[:51]+'...') if len(url)>51 else url.ljust(52)
    print(f"  {i:<3} {short:<52} {exp_name:<12} {pred_name:<12} {prob:>6.1f}%  {match}")

print(f"\n  Result: {correct}/{len(demo_urls)} correct ({correct/len(demo_urls)*100:.0f}% demo accuracy)")

---
## Pipeline Summary

| Step | Description | Output |
|------|-------------|--------|
| **1** | Dataset — 1,600 URLs, 20 features | `df.head()` + `df.shape` |
| **2** | 20 Features defined | Feature table + statistics |
| **3** | Pre-processing (drop URL, OHE tld, 80:20 split) | Train/test shapes |
| **4** | Random Forest (n_estimators=200) trained | Feature importances |
| **5** | Evaluated on unseen test data | Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix |
| **6** | Interactive UI — enter any URL | Phishing / Legitimate verdict + confidence + risk signals |

> **Tip:** In the interactive UI (Step 6 widget), try any real URL to get a live prediction with full feature breakdown.
